In [140]:
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pandas as pd 
import numpy as np
from pathlib import Path
import random
import gc 

import lightgbm as lgb
from catboost import CatBoostRanker, Pool
# from transformers import BertForSequenceClassification, AutoTokenizer

In [141]:
class Cfg: 
    emb_name = 'DeepPavlov/rubert-base-cased'
    editions_cols = ['edition_id', 'book_id', 'author_id', 'publication_year', 'age_restriction', 'language_id', 'publisher_id']

In [142]:
def seed_every_thing(seed):
    np.random.seed(seed)
    random.seed(seed)

seed_every_thing(42)

In [143]:
data_dir = Path('./data')
subm_dir = Path('./submit')

In [144]:
users = pd.read_csv(data_dir / 'users.csv') 
authors = pd.read_csv(data_dir / 'authors.csv')
genres = pd.read_csv(data_dir / 'genres.csv') 
book_genres = pd.read_csv(data_dir / 'book_genres.csv')
editions = pd.read_csv(data_dir / 'editions.csv')
interactions = pd.read_csv(data_dir / 'interactions.csv') 
targets = pd.read_csv(data_dir / 'targets.csv') 
candidates = pd.read_csv(data_dir / 'candidates.csv')
sample_submission = pd.read_csv(data_dir / 'example_submission.csv') 

# Users data

In [145]:
user_avgs = interactions.groupby('user_id')['rating'].agg(['mean', 'std', 'count']).reset_index()
user_avgs.columns = ['user_id', 'user_r_mean', 'user_r_std', 'user_r_count']

user_features = users.merge(user_avgs, on='user_id', how='left')
user_features['gender'] = user_features['gender'].fillna(-1)
user_features['age'] = user_features['age'].fillna(user_features.age.median())
user_features = user_features.fillna(0)

user_features.head()

,user_id,gender,age,user_r_mean,user_r_std,user_r_count
0,560,2.0,9.0,7.750000,1.502686,32
1,3410,2.0,39.0,8.571429,1.603567,14
2,5620,1.0,42.0,7.083333,1.311372,12
3,5700,2.0,40.0,9.142857,1.292412,14
4,6120,2.0,32.0,7.333333,1.352997,123


# Editions data

In [146]:
edition_features = editions[Cfg.editions_cols]

edition_avgs = interactions.groupby('edition_id')['rating'].agg(['mean', 'std', 'count']).reset_index()
edition_avgs.columns = ['edition_id', 'edition_r_mean', 'edition_r_std', 'edition_r_count']
edition_features = edition_features.merge(edition_avgs, on='edition_id', how='left')
edition_features = edition_features.fillna(0)

edition_popularity = interactions.groupby('edition_id').size().reset_index(name='edition_popularity')
edition_features = edition_features.merge(edition_popularity, on='edition_id', how='left')
edition_features = edition_features.fillna(0)

book_genre = book_genres.groupby('book_id')['genre_id'].first().reset_index()
edition_features = edition_features.merge(book_genre, on='book_id', how='left')

In [147]:
edition_features.head()

,edition_id,book_id,author_id,publication_year,age_restriction,language_id,publisher_id,edition_r_mean,edition_r_std,edition_r_count,edition_popularity,genre_id
0,1000000001,1093185,1.0,2000,18,119,9,0.0,0.0,0.0,0.0,424
1,1000000016,1548860,18.0,2005,16,119,1276,0.0,0.0,0.0,1.0,459
2,1000000020,20,21.0,2005,16,119,432,0.0,0.0,0.0,2.0,433
3,1000000035,35,33.0,2002,18,119,122,0.0,0.0,0.0,0.0,141
4,1000000070,891990,226891.0,2001,12,119,9,0.0,0.0,0.0,0.0,1235


# Prepare data

In [148]:
interactions['event_ts'] = pd.to_datetime(interactions['event_ts'])
max_date = interactions.event_ts.max()
split_date = max_date - pd.Timedelta(days=35)

train_part = interactions #[interactions.event_ts < split_date] 
#val_part = interactions[interactions.event_ts >= split_date]

In [149]:
def generate_ranker_data(interactions_df, editions_df, candidates_df=None, neg_per_pos=3, max_neg=30):
    """
    Generate ranking data: positives from interactions, negatives sampled.
    Relevance: 3 for any interaction (we don't use event_type!)
    """
    user_positives = interactions_df.groupby('user_id')['edition_id'].apply(set).to_dict()
    all_editions = set(editions_df['edition_id'].unique())
    
    rows = []
    
    for uid in tqdm(interactions_df['user_id'].unique()):
        positives = user_positives.get(uid, set())
        
        # Positives: relevance = 3
        for eid in positives:
            rows.append({'user_id': uid, 'edition_id': eid, 'relevance': 3})
        
        # Negatives: sample from non-interacted items
        if candidates_df is not None:
            user_cands = set(candidates_df[candidates_df['user_id'] == uid]['edition_id'])
            neg_pool = user_cands - positives
        else:
            neg_pool = all_editions - positives
        
        n_neg = min(max_neg, len(positives) * neg_per_pos, len(neg_pool))
        print(max_neg, len(positives) * neg_per_pos, len(neg_pool))
        if n_neg > 0 and len(neg_pool) > 0:
            negatives = np.random.choice(list(neg_pool), size=n_neg, replace=False)
            for eid in negatives:
                rows.append({'user_id': uid, 'edition_id': eid, 'relevance': 0})
    
    return pd.DataFrame(rows)

In [153]:
train_df = generate_ranker_data(train_part, editions, candidates)
print(f'Train samples: {len(train_df)}')
#val_df = generate_ranker_data(val_part, editions, candidates)
#print(f'Val samples: {len(val_df)}')

Generating cooc ranker data:   0%|          | 0/5067 [00:00<?, ?it/s]

Train samples: 378227


In [154]:
def add_features(df): 
    df = df.merge(edition_features, on='edition_id', how='left')
    df = df.merge(user_features, on='user_id', how='left')
    return df.fillna(-1)

In [155]:
train_df = add_features(train_df) 
val_df = add_features(val_df) 

In [156]:
# Define features
exclude_cols = ['relevance', 'user_id', 'edition_id']
feature_cols = [col for col in train_df.columns if col not in exclude_cols]

cat_cols = ['book_id', 'author_id', 'language_id', 'publisher_id', 'genre_id', 'gender']
cat_cols = [col for col in train_df.columns if col in cat_cols]

# Train Rankers

In [158]:
train_df_cbm = train_df.copy()
#val_df_cbm = val_df.copy()

for col in cat_cols: 
    train_df_cbm[col] = train_df_cbm[col].astype(str)
    #val_df_cbm[col] = val_df_cbm[col].astype(str)
    
train_pool = Pool(
    data=train_df_cbm[feature_cols],
    label=train_df_cbm['relevance'],
    group_id=train_df_cbm['user_id'],
    cat_features=cat_cols
)

#val_pool = Pool(
#    data=val_df_cbm[feature_cols],
#    label=val_df_cbm['relevance'],
#    group_id=val_df_cbm['user_id'],
#    cat_features=cat_cols
#)

In [159]:
cbm = CatBoostRanker(
    max_depth=8,
    loss_function='YetiRank',
    eval_metric='NDCG:top=20',
    iterations=300,
    early_stopping_rounds=50,
    #verbose=50,
    random_state=42
)

In [160]:
gc.collect()
cbm.fit(train_pool) #, eval_set=[val_pool])

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 612ms	remaining: 3m 2s
1:	total: 954ms	remaining: 2m 22s
2:	total: 1.26s	remaining: 2m 4s
3:	total: 1.54s	remaining: 1m 53s
4:	total: 1.87s	remaining: 1m 50s
5:	total: 2.11s	remaining: 1m 43s
6:	total: 2.33s	remaining: 1m 37s
7:	total: 2.6s	remaining: 1m 35s
8:	total: 2.87s	remaining: 1m 32s
9:	total: 3.15s	remaining: 1m 31s
10:	total: 3.42s	remaining: 1m 29s
11:	total: 3.66s	remaining: 1m 27s
12:	total: 3.9s	remaining: 1m 26s
13:	total: 4.16s	remaining: 1m 24s
14:	total: 4.4s	remaining: 1m 23s
15:	total: 4.63s	remaining: 1m 22s
16:	total: 4.88s	remaining: 1m 21s
17:	total: 5.09s	remaining: 1m 19s
18:	total: 5.31s	remaining: 1m 18s
19:	total: 5.54s	remaining: 1m 17s
20:	total: 5.79s	remaining: 1m 16s
21:	total: 6.02s	remaining: 1m 16s
22:	total: 6.27s	remaining: 1m 15s
23:	total: 6.48s	remaining: 1m 14s
24:	total: 6.71s	remaining: 1m 13s
25:	total: 6.95s	remaining: 1m 13s
26:	total: 7.21s	remaining: 1m 12s
27:	total: 7.44s	rema

In [161]:
importance = cbm.get_feature_importance(type='PredictionValuesChange')
feat_imp = pd.DataFrame({'feature': feature_cols, 'importance': importance})
print("CatBoost feature importance:")
feat_imp.sort_values('importance', ascending=False)

CatBoost feature importance:


,feature,importance
9,edition_popularity,30.338906
10,genre_id,12.065568
1,author_id,11.739658
6,edition_r_mean,7.116962
5,publisher_id,7.047252
2,publication_year,5.927779
12,age,3.516935
13,user_r_mean,3.384678
15,user_r_count,3.299560
3,age_restriction,3.030885


In [163]:
train_df_lgb = train_df.copy()
#val_df_lgb = val_df.copy()

for col in cat_cols:
    train_df_lgb[col] = train_df_lgb[col].astype('category')
    #val_df_lgb[col] = val_df_lgb[col].astype('category')

train_groups_lgb = train_df_lgb.groupby('user_id').size().values
#val_groups_lgb = val_df_lgb.groupby('user_id').size().values

In [164]:
train_lgb = lgb.Dataset(
    train_df_lgb[feature_cols],
    label=train_df_lgb['relevance'],
    group=train_groups_lgb,
    categorical_feature=cat_cols
)

#val_lgb = lgb.Dataset(
#    val_df_lgb[feature_cols],
#    label=val_df_lgb['relevance'],
#    group=val_groups_lgb,
#    reference=train_lgb,
#    categorical_feature=cat_cols
#)

In [165]:
lgb_params = {
    'objective': 'lambdarank',
    #'metric': 'ndcg',
    #'ndcg_eval_at': [20],
    'learning_rate': 0.05,
    'max_depth': 8,
    'num_leaves': 32,
    'l2_leaf_reg': 6,
    'l1_leaf_reg': 0.1,
    'min_data_in_leaf': 100,
    'verbose': -1,
    'seed': 42
}

lgb_model = lgb.train(
    lgb_params,
    train_lgb,
    num_boost_round=300,
    #valid_sets=[val_lgb],
    #valid_names=['val'],
    #callbacks=[
        #lgb.early_stopping(stopping_rounds=50),
        #lgb.log_evaluation(period=50)
    #]
)

In [166]:
# LightGBM feature importance
lgb_imp = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_model.feature_importance(importance_type='gain')
})
print("LightGBM feature importance:")
lgb_imp.sort_values('importance', ascending=False)

LightGBM feature importance:


,feature,importance
9,edition_popularity,354230.149295
0,book_id,11159.141465
15,user_r_count,4055.678223
1,author_id,2178.861615
11,gender,1692.952470
12,age,812.419205
14,user_r_std,685.253814
13,user_r_mean,628.869772
8,edition_r_count,556.210875
6,edition_r_mean,272.905163


# Generate Submission

In [167]:
# Prepare candidates
cands = candidates.copy()
cands = add_features(cands)

# CatBoost predictions
cands_cat = cands.copy()
for col in cat_cols:
    cands_cat[col] = cands_cat[col].astype(str)
score_cbm = cbm.predict(cands_cat[feature_cols])

# LightGBM predictions
cands_lgb = cands.copy()
for col in cat_cols:
    cands_lgb[col] = cands_lgb[col].astype('category')
score_lgb = lgb_model.predict(cands_lgb[feature_cols])

print(f"CatBoost scores: min={score_cbm.min():.3f}, max={score_cbm.max():.3f}")
print(f"LightGBM scores: min={score_lgb.min():.3f}, max={score_lgb.max():.3f}")

CatBoost scores: min=-5.479, max=5.971
LightGBM scores: min=-6.590, max=9.396


In [168]:
# Normalize scores to [0, 1] per user for fair blending
cands['score_cat'] = score_cbm
cands['score_lgb'] = score_lgb

# Normalize per user
for score_col in ['score_cat', 'score_lgb']:
    cands[score_col + '_norm'] = cands.groupby('user_id')[score_col].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-8)
    )

In [169]:
W_CAT = 0.9  # CatBoost
W_LGB = 0.1  # LightGBM

cands['score_blend'] = (
    W_CAT * cands['score_cat_norm'] + 
    W_LGB * cands['score_lgb_norm']
)

In [170]:
# Generate submission: top-20 per user by blended score
target_users = targets['user_id'].unique()

submission = cands[cands['user_id'].isin(target_users)][['user_id', 'edition_id', 'score_blend']]
submission = submission.sort_values(['user_id', 'score_blend'], ascending=[True, False])
submission = submission.groupby('user_id').head(20).reset_index(drop=True)
submission['rank'] = submission.groupby('user_id').cumcount() + 1
submission = submission[['user_id', 'edition_id', 'rank']]

print(f"Submission shape: {submission.shape}")
print(f"Users: {submission['user_id'].nunique()}")
print(f"Items per user: {submission.groupby('user_id').size().unique()}")

Submission shape: (101340, 3)
Users: 5067
Items per user: [20]


In [171]:
submission.to_csv(subm_dir / 'submission.csv', index=False)
submission.head(25)

,user_id,edition_id,rank
0,560,1010042716,1
1,560,1000019011,2
2,560,1010045137,3
3,560,1012838464,4
4,560,1006162723,5
5,560,1000887264,6
6,560,1000723978,7
7,560,1008525763,8
8,560,1010627534,9
9,560,1001285446,10
